# Fairall 9 from the VizieR SED service

This notebook queries the broadband photometry of the Seyfert 1 galaxy Fairall 9 from the [CDS VizieR SED service](http://vizier.cds.unistra.fr/vizier/sed/), keeps the bands with known transmission curves, and fits the SED with a host-capture model that accounts for the different apertures of the surveys. The redshift is fixed to the literature value.

You need internet access and the DSPS SSP file `tempdata.h5` in the repository root.

In [ ]:
import numpy as np
import numpyro.distributions as dist
from astropy.table import Table

from jaxsedfit import (
    AGNConfig,
    FilterSet,
    FitConfig,
    GalaxyConfig,
    InferenceConfig,
    JAXSEDFit,
    LikelihoodConfig,
    Observation,
    PhotometryData,
    PriorConfig,
    load_filter_curves,
)
from jaxsedfit.vizier import query_vizier_sed

## 1. Query the photometry

`query_vizier_sed` maps VizieR filters onto `jaxsedfit` filter names, keeps the most precise measurement per filter, and attaches a nominal PSF FWHM for each survey.

In [ ]:
rows, sed_table, url = query_vizier_sed("Fairall 9", radius=3.0)
phot = Table(rows=rows)["grahsp_filter", "flux_mjy", "err_mjy", "psf_fwhm_arcsec", "catalog"]
phot

## 2. Host-capture model

The surveys measure the galaxy through very different apertures, from SDSS PSF magnitudes to WISE's 6-12″ beams. With `use_host_capture_model=True`, the model flux in band $i$ is

$$F_{\mathrm{model},i} = F_{\mathrm{AGN},i} + \eta_i F_{\mathrm{host},i},$$

where the unresolved AGN is always fully captured and the captured host fraction $\eta_i$ grows with the band's spatial scale $R_i$ (`aperture_diameter_arcsec` if given, otherwise `psf_fwhm_arcsec`):

$$\eta_i = \sigma\left[s\,(\log R_i - \log R_0)\right],$$

with a fitted turnover scale $R_0$ and slope $s$.

In [ ]:
config = FitConfig(
    observation=Observation(object_id="Fairall 9", redshift=0.04587),
    photometry=PhotometryData(
        filter_names=list(phot["grahsp_filter"]),
        fluxes=list(phot["flux_mjy"]),
        errors=list(phot["err_mjy"]),
        psf_fwhm_arcsec=list(phot["psf_fwhm_arcsec"]),
    ),
    filters=FilterSet(curves=load_filter_curves(list(phot["grahsp_filter"]))),
    galaxy=GalaxyConfig(dsps_ssp_fn="../tempdata.h5"),
    agn=AGNConfig(agn_type=1),
    likelihood=LikelihoodConfig(use_host_capture_model=True),
    inference=InferenceConfig(method="optax+nuts", map_steps=400, learning_rate=5e-3, num_warmup=100, num_samples=50),
    prior_config=PriorConfig(
        stellar_mass=dist.Normal(10.5, 1.0),
        host={"ebv_gal": dist.HalfNormal(0.15)},
        agn={"ebv_agn": dist.HalfNormal(0.15)},
    ),
)

fitter = JAXSEDFit(config)
result = fitter.fit(progress_bar=False)

## 3. Results

In [ ]:
fitter.plot_sed(show=True);

In [ ]:
pred = fitter.predict()
Table(
    {
        "filter": phot["grahsp_filter"],
        "observed": phot["flux_mjy"],
        "error": phot["err_mjy"],
        "model": pred["pred_fluxes"][0],
        "host": pred["host_fluxes"][0],
        "dust": pred["dust_fluxes"][0],
        "agn": pred["agn_fluxes"][0],
    }
)

## Notes

- To fit the redshift too, set `Observation(redshift_mode="fit", redshift_err=...)` or pass a photo-z prior (see the next tutorial).
- Use `method="optax"` for a quick MAP check, or increase `num_warmup` and `num_samples` for a fuller posterior.